# 02 Data Cleaning

This notebook cleans and prepares the electronics store ecommerce dataset for later analysis.

## 1. Import libraries and define file paths

We start by importing pandas and setting the input and output file paths for this project.

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("..") / "data" / "events.csv"
OUTPUT_PATH = Path("..") / "outputs" / "events_cleaned.csv"

print(f"Input file: {INPUT_PATH.resolve()}")
print(f"Output file: {OUTPUT_PATH.resolve()}")

## 2. Load the raw dataset

We read the raw CSV file from the `data` folder.

In [ ]:
df = pd.read_csv(INPUT_PATH)
original_shape = df.shape
print("Original shape:", original_shape)
df.head()

## 3. Convert `event_time` to datetime

This makes the time column easier to work with and allows us to create date-based fields later.

In [ ]:
df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
print("Missing event_time values after conversion:", df["event_time"].isna().sum())

## 4. Check for duplicate rows

We count duplicate rows first, then remove them so they do not inflate later metrics.

In [ ]:
duplicate_rows = df.duplicated().sum()
print("Duplicate rows found:", duplicate_rows)

df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)

## 5. Check missing or invalid values in key columns

Here we inspect the most important fields we will rely on later for analytics.

In [ ]:
key_columns = [
    "event_time",
    "event_type",
    "product_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

missing_key_values = df[key_columns].isna().sum().sort_values(ascending=False)
print("Missing values in key columns:")
print(missing_key_values)

print("\nUnique event types:")
print(sorted(df["event_type"].dropna().unique().tolist()))

invalid_user_ids = (df["user_id"] <= 0).sum()
invalid_product_ids = (df["product_id"] <= 0).sum()

print("\nInvalid user_id count (<= 0):", invalid_user_ids)
print("Invalid product_id count (<= 0):", invalid_product_ids)

## 6. Make sure `price` is numeric

If needed, we convert the column to numeric and check for missing or non-positive values.

In [ ]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")

missing_price_count = df["price"].isna().sum()
non_positive_price_count = (df["price"] <= 0).sum()

print("Missing price values after conversion:", missing_price_count)
print("Non-positive price values:", non_positive_price_count)

## 7. Clean the most important fields

For analytics, we keep rows with the essential identifiers and valid time values. We also handle missing category and brand values in a practical way by filling them with `Unknown`, which keeps those records available for summaries and dashboards.

In [ ]:
rows_before_cleaning = len(df)

df = df.dropna(subset=["event_time", "event_type", "product_id", "user_id", "price"]).copy()
df = df[df["price"] > 0].copy()

df["brand"] = df["brand"].fillna("Unknown")
df["category_code"] = df["category_code"].fillna("Unknown")
df["user_session"] = df["user_session"].fillna("missing_session")

rows_removed_during_cleaning = rows_before_cleaning - len(df)
print("Rows removed during cleaning:", rows_removed_during_cleaning)
print("Shape after cleaning:", df.shape)

## 8. Create derived date columns

These fields will make it easier to summarize the data by day, month, and week later.

In [ ]:
df["event_date"] = df["event_time"].dt.date
df["event_month"] = df["event_time"].dt.to_period("M").astype(str)
df["event_week"] = df["event_time"].dt.to_period("W").astype(str)

df[["event_time", "event_date", "event_month", "event_week"]].head()

## 9. Save the cleaned dataset

We save the cleaned file to the `outputs` folder so it can be reused in later notebooks.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH.resolve()}")

## 10. Final cleaning summary

This short summary shows the final dataset size, how many duplicate rows were removed, what missing values remain in key columns, and where the cleaned file was saved.

In [ ]:
remaining_missing = df[key_columns].isna().sum()

print("Final shape:", df.shape)
print("Duplicate rows removed:", duplicate_rows)
print("Missing values remaining in key columns:")
print(remaining_missing)
print("Output file path:", OUTPUT_PATH.resolve())